# AstroStylist — Qwen-Image-Edit glow-up server (free Kaggle GPU)

Runs **Qwen-Image-Edit-2511** on Kaggle's free GPU and exposes it as a public API your local AstroStylist app can call.

**Before you run:**
1. Right sidebar → **Settings → Accelerator → GPU T4 x2** (or P100).
2. Right sidebar → **Settings → Internet → On** (required for downloads + the tunnel).
3. Run each cell top to bottom. The **last cell prints a public URL** — paste it into `backend/.env` as `REMOTE_GLOWUP_URL=...`, then restart your CV service.

Keep this notebook tab **open** while you use the app — when the kernel stops, the URL dies.

> First run downloads ~20–40 GB of weights (slow, one-time per session). Each glow-up then takes ~1–3 min on a free T4.

In [ ]:
# 1) Install deps + cloudflared (the tunnel that gives us a public URL)
!pip install -q -U diffusers transformers accelerate safetensors sentencepiece fastapi "uvicorn[standard]" nest_asyncio pillow
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("deps + cloudflared ready")

In [ ]:
# 2) Load Qwen-Image-Edit-2511. T4/P100 have no bf16 → use float16 + cpu-offload so a 20B model fits in 16GB.
import torch
from diffusers import QwenImageEditPlusPipeline

MODEL = "Qwen/Qwen-Image-Edit-2511"
pipe = QwenImageEditPlusPipeline.from_pretrained(MODEL, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()          # streams modules GPU<->CPU so it fits ~16GB (slower, but fits)
pipe.set_progress_bar_config(disable=True)
print("pipeline loaded")

# --- If this cell OOMs on a single T4, use this lighter offload instead (comment the two lines above):
# pipe = QwenImageEditPlusPipeline.from_pretrained(MODEL, torch_dtype=torch.float16)
# pipe.enable_sequential_cpu_offload()   # fits in less VRAM, noticeably slower per image

In [ ]:
# 3) The API: POST /glowup {user_image (data URI/base64), prompt, negative_prompt} -> {success, image (data URI)}
import base64, io, re
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="AstroStylist Qwen glow-up")

class GlowIn(BaseModel):
    user_image: str
    prompt: str
    negative_prompt: str = " "

def _decode(u):
    m = re.match(r"data:[^;]+;base64,(.*)$", u, re.S)
    b64 = m.group(1) if m else u
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")

@app.get("/glowup/status")
def status():
    return {"ready": True, "model": MODEL, "device": "kaggle"}

@app.post("/glowup")
def glowup(inp: GlowIn):
    try:
        img = _decode(inp.user_image)
        img.thumbnail((1024, 1024))       # cap size -> less VRAM, faster
        with torch.inference_mode():
            out = pipe(
                image=[img],
                prompt=inp.prompt,
                negative_prompt=inp.negative_prompt or " ",
                true_cfg_scale=4.0,
                num_inference_steps=30,
                guidance_scale=1.0,
                num_images_per_prompt=1,
                generator=torch.manual_seed(0),
            ).images[0]
        buf = io.BytesIO(); out.save(buf, format="PNG")
        data = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
        return {"success": True, "image": data, "renderer": MODEL, "device": "kaggle"}
    except Exception as e:
        return {"success": False, "error": f"{type(e).__name__}: {e}", "fallback_available": True}

print("API defined")

In [ ]:
# 4) Serve it + open a public tunnel. Copy the printed URL into backend/.env as REMOTE_GLOWUP_URL=...
import nest_asyncio, uvicorn, threading, subprocess, time, re
nest_asyncio.apply()

def _serve():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=_serve, daemon=True).start()
time.sleep(4)

logf = open("/tmp/cf.log", "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                 stdout=logf, stderr=subprocess.STDOUT)

url = None
for _ in range(40):
    time.sleep(2)
    txt = open("/tmp/cf.log").read()
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", txt)
    if m:
        url = m.group(0); break

print("\n" + "="*60)
print("PUBLIC URL:", url or "(not found — re-run this cell)")
print("Put this in backend/.env, then restart the CV service:")
print("   REMOTE_GLOWUP_URL=" + (url or "<url>"))
print("="*60)

In [ ]:
# 5) (optional) Smoke-test the server from inside the notebook before wiring the app.
import requests, base64
# tiny 2x2 test image
px = base64.b64encode(bytes.fromhex("89504e470d0a1a0a")).decode()  # not a real image; replace to truly test
print(requests.get("http://localhost:8000/glowup/status").json())